## 🔧 2. Agents with Local Tools
**Goal**: Enhance agents with local tools — custom Python functions that run locally while the agent schema is registered server-side in Foundry with full **tracing**.

- `@tool` decorator — define local tool implementations
- `FunctionTool` schema — register tool schemas in Foundry
- `FoundryAgent(tools=[...])` — bind local implementations to the agent
- **Tracing** — all interactions visible in Foundry portal

---
### Setup — Load Environment & Connect to Foundry

In [1]:
# ── Load environment & configuration ─────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath("..") if "notebooks" in os.getcwd() else ".")

from dotenv import load_dotenv
load_dotenv()

from utils.config import get_env, get_credential
from agent_framework import tool

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    PromptAgentDefinition,
    FunctionTool as AzureFunctionTool,
)
from agent_framework.foundry import FoundryAgent

# Endpoints
PROJECT_ENDPOINT = get_env("AZURE_AI_PROJECT_ENDPOINT_FULL")
MODEL = get_env("AZURE_AI_MODEL_DEPLOYMENT_NAME")

project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=get_credential(),
)

# ── Utility: Convert @tool-decorated FunctionTool to AzureFunctionTool ─────────
def to_azure_function_tool(tool_func, strict=False):
    """Convert a @tool-decorated FunctionTool to AzureFunctionTool for Foundry.
    
    Args:
        tool_func: A function decorated with @tool
        strict: Whether to enforce strict schema validation (default: False)
    
    Returns:
        AzureFunctionTool with auto-extracted name, description, and parameters
    """
    return AzureFunctionTool(
        name=tool_func.name,
        description=tool_func.description,
        parameters=tool_func.parameters(),
        strict=strict,
    )


print(f"📡 Project endpoint: {PROJECT_ENDPOINT}")
print(f"🧠 Model: {MODEL}")
print("✅ Connected to Foundry")

c:\Dibyendu\Code\agent-framework-demo-py\.venv\Lib\site-packages\agent_framework\_skills.py:116: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
c:\Dibyendu\Code\agent-framework-demo-py\.venv\Lib\site-packages\agent_framework\_harness\_memory.py:651: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.


📡 Project endpoint: https://foundry-demo-us-east-resource.services.ai.azure.com/api/projects/prj-foundry-demo-us-east-dev-001
🧠 Model: gpt-4o
✅ Connected to Foundry


---
### A. Simple Tool Example — Weather

In [2]:
# ── Simple weather tool demo ──────────────────────────────────────────────────
from typing import Annotated
from pydantic import Field

# Register a simple weather agent in Foundry
WEATHER_AGENT_NAME = "WeatherAssistant-af-py"

@tool(description="Get the weather for a given location.")
def get_weather(
    location: Annotated[str, Field(description="The location to get the weather for.")],
) -> str:
    """Simulated weather lookup."""
    return f"The weather in {location} is cloudy with a high of 15°C."

weather_agent_version = project.agents.create_version(
    agent_name=WEATHER_AGENT_NAME,
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions="You are a helpful assistant that can get weather information.",
        tools=[to_azure_function_tool(get_weather)],
    ),
)

weather_agent = FoundryAgent(
    project_endpoint=PROJECT_ENDPOINT,
    agent_name=weather_agent_version.name,
    agent_version=str(weather_agent_version.version),
    credential=get_credential(),
    tools=[get_weather],
)

result = await weather_agent.run("What is the weather like in Amsterdam?")
print(f"🗨️  {result}")

2026-06-01 19:50:18,825 - azure.ai.agentserver - INFO - Application Insights not configured, telemetry export disabled.
🗨️  The weather in Amsterdam is currently cloudy with a high of 15°C.


---
### B. Another example with multiple function tools 🛠️

In [3]:
# ── D1. Define custom function tools ─────────────────────────────────────────

@tool(description="Look up current inventory count for a product SKU.")
def check_inventory(
    sku: Annotated[str, Field(description="The product SKU, e.g. CW-PRO-001")],
) -> str:
    """Simulated inventory lookup."""
    inventory = {
        "CW-PRO-001": 142,
        "CW-LITE-002": 580,
        "CC-CLOUD-003": 999,
    }
    count = inventory.get(sku.upper(), None)
    if count is None:
        return f"SKU '{sku}' not found in inventory."
    return f"SKU {sku}: {count} units in stock."


@tool(description="Submit a support ticket with a description.")
def create_support_ticket(
    title: Annotated[str, Field(description="Short summary of the issue")],
    description: Annotated[str, Field(description="Detailed description")],
) -> str:
    """Simulated ticket creation."""
    import random
    ticket_id = f"TKT-{random.randint(1000, 9999)}"
    return f"✅ Ticket {ticket_id} created: '{title}'"


print("🛠️  Custom tools defined: check_inventory, create_support_ticket")

🛠️  Custom tools defined: check_inventory, create_support_ticket


---
### Create Agent in Foundry with tool definition

In [4]:
# ── D2. Register Function Tool agent in Foundry ──────────────────────────────
TOOL_AGENT_NAME = "ContosoHelper-af-py"

# Define function schemas for the Foundry agent definition
tool_agent_version = project.agents.create_version(
    agent_name=TOOL_AGENT_NAME,
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions=(
            "You are a Contoso support agent. Use the available tools to help "
            "customers with inventory queries and support requests."
        ),
        tools=[
            to_azure_function_tool(check_inventory),
            to_azure_function_tool(create_support_ticket),
        ],
    ),
)

print(f"✅ Created agent version: {tool_agent_version.name} v{tool_agent_version.version}")
print("ℹ️  Use the next cell to bind local tools and run the agent.")

✅ Created agent version: ContosoHelper-af-py v1
ℹ️  Use the next cell to bind local tools and run the agent.


---
### Run the Agent by providing local function tools

In [5]:
# ── D3. Connect via Agent Framework with local tool implementations ───────────
tool_agent = FoundryAgent(
    project_endpoint=PROJECT_ENDPOINT,
    agent_name=tool_agent_version.name,
    agent_version=str(tool_agent_version.version),
    credential=get_credential(),
    tools=[check_inventory, create_support_ticket],
)

print(f"✅ Loaded agent: {tool_agent_version.name} v{tool_agent_version.version}")
print("🤖 FoundryAgent bound with local tools and ready.")

# Test inventory lookup
inv_result = await tool_agent.run("How many Widget Pro units (CW-PRO-001) do we have in stock?")
print(f"\n🗨️  {inv_result}")

# Test ticket creation
ticket_result = await tool_agent.run(
    "Create a support ticket: 'Cloud Connector sync failing in EU region' "
    "with description 'OTA updates timing out after 30 seconds for EU customers.'"
)
print(f"\n🗨️  {ticket_result}")

print(f"\n🔍 Check Foundry portal → Tracing to see these interactions!")

✅ Loaded agent: ContosoHelper-af-py v1
🤖 FoundryAgent bound with local tools and ready.

🗨️  We currently have 142 units of the Widget Pro (SKU: CW-PRO-001) in stock.

🗨️  The support ticket "Cloud Connector sync failing in EU region" has been successfully created with the ID **TKT-1646**. The issue regarding OTA updates timing out for EU customers is now under review.

🔍 Check Foundry portal → Tracing to see these interactions!


---
### Cleanup 🧹

In [6]:
# ── Cleanup — Delete agents from Foundry ─────────────────────────────────────
for name in [WEATHER_AGENT_NAME, TOOL_AGENT_NAME]:
    try:
        project.agents.delete(agent_name=name)
        print(f"🧹 Deleted agent: {name}")
    except Exception as e:
        print(f"⚠️  Could not delete {name}: {e}")

print("✅ Cleanup complete")

🧹 Deleted agent: WeatherAssistant-af-py
🧹 Deleted agent: ContosoHelper-af-py
✅ Cleanup complete


---
### ✅ What you demonstrated
| Capability | How |
|---|---|
| Simple local tool | `@tool` decorator + `FoundryAgent(tools=[...])` |
| Inventory lookup | `check_inventory` — local function, schema in Foundry |
| Ticket creation | `create_support_ticket` — local function, schema in Foundry |
| Auto tool selection | Agent autonomously picks the right tool |
| Server-side tracing | All interactions visible in Foundry portal → Tracing |

**Next →** [03_agent_tools_builtin.ipynb](./03_agent_tools_builtin.ipynb) — Foundry Built in tools.